In [ ]:
# !python -m spacy download en_core_web_sm
# !python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 27.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 34.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install datasets
!pip install sentencepiece
!pip install evaluate
!pip install bert_score
!pip install accelerate -U
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 26.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 726.7 kB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-m

In [ ]:
import os
import torch
import datasets
import spacy
import numpy as np
from evaluate import load
from torch.utils.data import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
path = "/content/drive/MyDrive/NLP_2024/Project/Data_after_cleaning_and_splits/"
os.chdir(path)

In [ ]:
# spacy.load('en_core_web_sm')
# spacy.load('de_core_news_sm')

In [ ]:
train_dataset = pd.read_csv('train_dataset_final.csv')
test_dataset = pd.read_csv('test_dataset_final.csv')
val_dataset = pd.read_csv('val_dataset_final.csv')
train_dataset = train_dataset[train_dataset['offensiveYN']==1.0]
test_dataset = test_dataset[test_dataset['offensiveYN']==1.0]
val_dataset = val_dataset[val_dataset['offensiveYN']==1.0]

In [ ]:
print(len(train_dataset))
print(len(test_dataset))
print(len(val_dataset))

10974
1750
1620


In [ ]:
train_dataset = train_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
test_dataset = test_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
val_dataset = val_dataset[['post','targetMinority','targetCategory','rationale','targetStereotype']]
train_dataset = train_dataset.dropna()
test_dataset = test_dataset.dropna()
val_dataset = val_dataset.dropna()

In [ ]:
print(len(train_dataset))
print(len(test_dataset))
print(len(val_dataset))

10974
1750
1620


In [ ]:
test_dataset

,post,targetMinority,targetCategory,rationale,targetStereotype
1,When u hitting it from the back u gotta call h...,women,gender,This post promotes the use of degrading langua...,sexually promiscuous women are called sluts
3,You niggas talking to these females trying to ...,women,gender,This post displays a clear disdain and contemp...,women are self-centered/egotistical
5,A Nigguh spitting dat crack to your bitch,black folks,race,This post is considered hateful because it pro...,black people are crackheads
6,RT @lesxo_: cant turn a hoe into a husband either,"women, trans men",gender,This post is considered hateful because it imp...,women who have sex before marriage are ruined.
7,"""@ReachtheeCLIMAX: bitches can't cook but know...",women,gender,This post is hateful because it objectifies an...,women are used for sex
...,...,...,...,...,...
3784,Female depression and Jon Snow have one thing ...,women,gender,This post is hateful because it is making an i...,women can’t be depressed
3786,A white male toddler needs to have its head sm...,"black folks, latino/latina folks",race,This post promotes violence and hate towards a...,blacks are niggers
3787,women are gay\nwomen are all lesbians and that...,"women, lesbian women",gender,This post is hateful because it displays a cle...,women hate men
3796,If you became chad tier good looking through s...,women,gender,"This post shows a clear desire for revenge, wh...",deserve revenge for rejecting men


In [ ]:
# Decode translations
# test_translations = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [ ]:
from transformers import AutoTokenizer

t5_checkpoint = "google-t5/t5-small"
finetune_tokenizer = AutoTokenizer.from_pretrained(t5_checkpoint)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/google-t5/t5-small/resolve/main/tokenizer.json: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out.
Trying to resume download...
Trying to resume download...


tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
task_prefix = "generate implication for this hateful post "
task_mid1 = " with target minority "
task_mid2 = " with target category "
task_mid3 = " with rationale "
# post targetMinority targetCategory rationale
# targetStereotype

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, data,tokenizer):
    # col names ['post','targetMinority','targetCategory','rationale','targetStereotype']
    self.posts = data['post'].tolist()
    self.minority = data['targetMinority'].tolist()
    self.category = data['targetCategory'].tolist()
    self.rationale = data['rationale'].tolist()
    self.label = data['targetStereotype'].tolist()
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.posts)

  def __getitem__(self, idx):
    inputs = task_prefix + str(self.posts[idx])+ task_mid1+str(self.minority[idx])+ task_mid2+str(self.category[idx]) +task_mid3+ str(self.rationale[idx])
    targets = str(self.label[idx])
    input_enc = self.tokenizer(inputs, text_target=targets, max_length=512, padding=True ,truncation=True)

    # print(input_enc)
    return {
          'input_ids': input_enc['input_ids'],
          'attention_mask': input_enc['attention_mask'],
          'labels':input_enc['labels']
          }

In [ ]:
# tokenized_dataset_train = train_dataset.map(preprocess_function, batched=True)
tokenized_dataset_train = CustomDataset(train_dataset,finetune_tokenizer)

In [ ]:
c = 0
for i in tokenized_dataset_train:
  print(i)
  print(len(i['input_ids']))
  if c>2:
    break
  c+=1
# just using to see length of input ids

{'input_ids': [3806, 3, 28722, 21, 48, 5591, 1329, 442, 3, 5934, 3320, 23, 2703, 956, 32, 10, 3, 13076, 12417, 3, 89, 4636, 53, 3, 7315, 1304, 312, 22780, 29, 5, 148, 8882, 5341, 13721, 19126, 2293, 29, 63, 21622, 3, 89, 9, 122, 10779, 5, 28, 2387, 16392, 1001, 5265, 28, 2387, 3295, 1964, 28, 12226, 15, 100, 442, 19, 5591, 1329, 250, 34, 2284, 20, 3822, 6546, 1612, 11, 3, 7, 40, 3589, 12, 3211, 312, 22780, 29, 2549, 6, 3, 9, 8304, 1001, 17893, 5, 37, 169, 13, 1353, 224, 38, 96, 7315, 1304, 976, 96, 22498, 109, 2293, 29, 63, 976, 11, 96, 2157, 4397, 121, 33, 7447, 12130, 11, 20, 12450, 2610, 1587, 1001, 151, 5, 5433, 6, 8, 442, 92, 2284, 8, 1448, 96, 89, 9, 122, 10779, 121, 38, 46, 21548, 6, 31324, 1014, 13503, 27426, 5, 37, 1879, 5739, 13, 8, 442, 19, 8299, 11, 3353, 28, 11213, 6, 2924, 3, 9, 964, 8762, 12, 6263, 11, 36, 26192, 8, 568, 2799, 5, 100, 686, 13, 1612, 11, 13972, 13, 3, 9, 806, 1964, 11, 6949, 12602, 19, 29452, 11, 2519, 7, 5591, 11, 9192, 257, 5, 1], 'attention_mask': [1, 

In [ ]:
# tokenized_dataset_val = val_dataset.map(preprocess_function, batched=True)
tokenized_dataset_val = CustomDataset(val_dataset,finetune_tokenizer)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=finetune_tokenizer, model=t5_checkpoint)

In [ ]:
metric = load("sacrebleu")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

finetune_model = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="finetuned_T5_final",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=True
)

trainer = Seq2SeqTrainer(
    model=finetune_model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_val,
    tokenizer=finetune_tokenizer,
    data_collator=data_collator,
)

In [ ]:
# train_results = trainer.train()

Epoch,Training Loss,Validation Loss
1,2.034800,1.970111
2,2.019800,1.933984
3,1.956800,1.899825
4,1.944800,1.880520
5,1.924600,1.853415
6,1.852400,1.834922
7,1.844000,1.822495
8,1.834200,1.805177
9,1.793300,1.796420
10,1.780600,1.785535


In [ ]:
# trainer.save_model()

In [ ]:
# finetune_tokenizer.save_pretrained("")

TypeError: PreTrainedTokenizerBase.save_pretrained() missing 1 required positional argument: 'save_directory'

In [ ]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=c646f809d92e91ec47751c5c1d8fe2b7115459ec746edd5aaf873b8f419b3e88
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
finetune_model = AutoModelForSeq2SeqLM.from_pretrained('finetuned_T5_final')

In [ ]:
finetune_model.to(DEVICE)

In [ ]:
test_posts = test_dataset['post'].tolist()
test_minority = test_dataset['targetMinority'].tolist()
test_category = test_dataset['targetCategory'].tolist()
test_rationale = test_dataset['rationale'].tolist()
test_label = test_dataset['targetStereotype'].tolist()

In [ ]:
test_targets = []
test_generated = []
i = 0
for idx in range(len(test_posts)):
    inputs = task_prefix + str(test_posts[idx])+ task_mid1+str(test_minority[idx])+ task_mid2+str(test_category[idx]) +task_mid3+ str(test_rationale[idx])
    input_ids = finetune_tokenizer.encode(inputs, return_tensors="pt").to(DEVICE)
    output = finetune_model.generate(input_ids, max_length=50, num_beams=4, early_stopping=True)
    output_text = finetune_tokenizer.decode(output[0], skip_special_tokens=True)
    test_generated.append(output_text)
    test_targets.append(test_label[idx])

In [ ]:
print(test_targets[500])
print(test_generated[500])

white folks are animals
white people are less than human


In [ ]:
!pip install rouge-score

In [ ]:
import evaluate
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=test_generated, references=test_targets)
print(results)

{'rouge1': 0.3928743448992891, 'rouge2': 0.2279708402708399, 'rougeL': 0.391570649304993, 'rougeLsum': 0.3914604659505806}


In [ ]:
from datasets import load_metric
metric = load_metric("rouge")
results = metric.compute(predictions=test_generated, references=test_targets)
# print(results)
result = list(results.items())
for index in range(len(result)-1):
  type_, scores = result[index]
  precision = scores.mid.precision
  recall = scores.mid.recall
  f1 = scores.mid.fmeasure
  print(str(type_)+" with precision "+str(precision)+" with recall "+str(recall)+" and F1 score "+str(f1))

/usr/local/lib/python3.10/dist-packages/datasets/load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


rouge1 with precision 0.4117227891156462 with recall 0.39391515310086783 and F1 score 0.3928743448992891
rouge2 with precision 0.23781190476190459 with recall 0.2292044217687073 and F1 score 0.2279708402708399
rougeL with precision 0.4104119047619049 with recall 0.3930772213314231 and F1 score 0.391570649304993


In [ ]:
from bert_score import score as bert_score
Ptestft, Rtestft, F1testft = bert_score(cands=test_generated, refs=test_targets, lang='en', verbose=True)
print("BERTScore Precision:", Ptestft.mean().item())
print("BERTScore Recall:", Rtestft.mean().item())
print("BERTScore F1:", F1testft.mean().item())

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/28 [00:00<?, ?it/s]

done in 5.58 seconds, 313.43 sentences/sec
BERTScore Precision: 0.9064441919326782
BERTScore Recall: 0.9071900248527527
BERTScore F1: 0.9066716432571411
